[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/26_recommender_systems/90_exercises.ipynb)

# Bài tập — Chương 26: Recommender Systems (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/26_recommender_systems

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import metrics as M
from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

ratings = pd.read_csv(repo_path("data", "sample", "ratings.csv"))
items = pd.read_csv(repo_path("data", "sample", "items.csv"))
n_users, n_items = ratings.user_id.max() + 1, ratings.item_id.max() + 1


def split_per_user(ratings, test_frac=0.2, min_ratings=5, seed=0):
    rng = np.random.default_rng(seed)
    test_mask = np.zeros(len(ratings), bool)
    for _, g in ratings.groupby("user_id"):
        if len(g) >= min_ratings:
            test_mask[rng.choice(g.index, max(1, int(test_frac * len(g))), replace=False)] = True
    return ratings[~test_mask], ratings[test_mask]


train, test = split_per_user(ratings)
u_tr, i_tr, y_tr = train.user_id.to_numpy(), train.item_id.to_numpy(), train.rating.to_numpy(float)
u_te, i_te, y_te = test.user_id.to_numpy(), test.item_id.to_numpy(), test.rating.to_numpy(float)
B = np.zeros((n_users, n_items))
B[u_tr, i_tr] = 1.0
seen = {u: set(g.item_id) for u, g in train.groupby("user_id")}
liked_test = {u: [i for i, r in zip(g.item_id, g.rating) if r >= 4] for u, g in test.groupby("user_id")}


def precision10(score_fn):
    """precision@10 trung bình trên các user có phim thích trong test (loại phim đã thấy trong train)."""
    out = []
    for u, liked in liked_test.items():
        if not liked:
            continue
        s = np.asarray(score_fn(u), float).copy()
        s[list(seen.get(u, []))] = -np.inf
        out.append(M.precision_at_k(np.argsort(-s)[:10], liked, 10))
    return float(np.mean(out))

## L0 — Nhớ khái niệm

**Câu 1.** (a) Phản hồi tường minh vs ngầm. (b) Content-based vs collaborative filtering: mỗi loại cần dữ liệu gì, xử lý cold start nào? (c) Hàm mục tiêu của matrix factorization cho rating. (d) Định nghĩa NDCG@k; vì sao có mẫu số $\log_2(i + 1)$? (e) "Missing not at random" trong dữ liệu gợi ý là gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một đội so sánh offline: MF có RMSE 0.61, tốt hơn mô hình hiện tại (gợi ý phim phổ biến theo tuần) — vốn "không có RMSE". Họ triển khai: mỗi user thấy 10 phim có rating dự đoán cao nhất. Tỉ lệ click giảm 30%. Giải thích và đề xuất cách đánh giá/đổi mô hình.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** (a) $\mu = 3.8$, $m = 10$. Phim A: 2 rating, trung bình 5.0. Phim B: 100 rating, trung bình 4.2. Tính trung bình giảm chấn; phim nào xếp trên? (b) Cột "đã xem" của hai phim trên 4 user: $i = [1, 1, 0, 1]$, $j = [1, 0, 0, 1]$. Cosine?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
damped_A = None
damped_B = None
cos_ij = None
### END CODE HERE ###
print(f"A = {damped_A:.4f} | B = {damped_B:.4f} | cos = {cos_ij:.4f}")

In [ ]:
check_close(damped_A, 4.0, message="trung bình giảm chấn A")
check_close(damped_B, 458 / 110, message="trung bình giảm chấn B")
check_greater(damped_B, damped_A, "B xếp trên A dù trung bình thô thấp hơn")
check_close(cos_ij, 2 / np.sqrt(6), message="cosine")

**Câu 4.** Gợi ý `[3, 1, 4, 7, 5]`; user thích (nhị phân) `{1, 5, 9}`. Tính precision@5, recall@5, MRR, AP@5, NDCG@5 (mức liên quan = 1 cho mọi item thích).

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
p5 = None
r5 = None
mrr4 = None
ap5 = None
ndcg5 = None
### END CODE HERE ###
print(f"P@5 = {p5:.3f} | R@5 = {r5:.3f} | MRR = {mrr4:.3f} | AP@5 = {ap5:.3f} | NDCG@5 = {ndcg5:.4f}")

In [ ]:
recs4, rel4 = [3, 1, 4, 7, 5], {1, 5, 9}
check_close(p5, M.precision_at_k(recs4, rel4, 5), message="precision@5")
check_close(r5, M.recall_at_k(recs4, rel4, 5), message="recall@5")
check_close(ap5, M.average_precision_at_k(recs4, rel4, 5), message="AP@5")
check_close(ndcg5, M.ndcg_at_k(recs4, {1: 1, 5: 1, 9: 1}, 5), message="NDCG@5")
check_close(mrr4, 0.5, message="MRR")

## L3 — Code

**Bài 5.** Cài đặt `recall_at_k`, `reciprocal_rank(recommended, relevant)` (1/vị trí item đúng đầu tiên, 0 nếu không có) và `ndcg_at_k(recommended, relevance, k)` (relevance: dict item → mức, gain $2^{rel} - 1$). So với `ml_course.metrics`.

In [ ]:
def recall_at_k(recommended, relevant, k):
    ### START CODE HERE ### (≈ 2 dòng)
    return None
    ### END CODE HERE ###


def reciprocal_rank(recommended, relevant):
    ### START CODE HERE ### (≈ 4 dòng)
    return None
    ### END CODE HERE ###


def ndcg_at_k(recommended, relevance, k):
    ### START CODE HERE ### (≈ 4 dòng)
    # DCG = Σ (2^rel − 1) / log2(i + 2) với i = 0..k−1; IDCG: sắp relevance giảm dần; trả về DCG / IDCG (0 nếu IDCG = 0)
    pass
    ### END CODE HERE ###

In [ ]:
rng5 = np.random.default_rng(5)
for _ in range(20):
    recs = list(rng5.permutation(30)[:10])
    rel = {int(i): int(rng5.integers(1, 4)) for i in rng5.choice(30, 5, replace=False)}
    check_close(recall_at_k(recs, rel, 10), M.recall_at_k(recs, rel, 10), message="recall@k")
    check_close(ndcg_at_k(recs, rel, 10), M.ndcg_at_k(recs, rel, 10), message="NDCG@k")
check_close(reciprocal_rank([3, 1, 4], {4, 1}), 0.5, message="MRR")
check_close(reciprocal_rank([3, 1, 4], {9}), 0.0, message="không có item đúng")

**Bài 6.** Cài đặt `FunkSVD(k, lr, reg, epochs, seed)` (MF có thiên lệch, huấn luyện bằng SGD như bài 03) với `fit(users, items, ratings)` và `predict(users, items)`. Train trên `train` với `k=20, lr=0.01, reg=0.1, epochs=40`; RMSE test (`rmse6`) phải tốt hơn baseline $\mu + b_u + b_i$ (~0.631).

In [ ]:
class FunkSVD:
    def __init__(self, k=20, lr=0.01, reg=0.1, epochs=40, seed=0):
        self.k, self.lr, self.reg, self.epochs, self.seed = k, lr, reg, epochs, seed

    def fit(self, users, items_, y):
        r = np.random.default_rng(self.seed)
        self.mu_ = float(np.mean(y))
        self.P_ = r.normal(0, 0.1, (n_users, self.k))
        self.Q_ = r.normal(0, 0.1, (n_items, self.k))
        self.bu_, self.bi_ = np.zeros(n_users), np.zeros(n_items)
        ### START CODE HERE ### (≈ 11 dòng)
        # mỗi epoch: duyệt các rating theo thứ tự ngẫu nhiên (r.permutation)
        # e = y − (mu + bu[u] + bi[i] + P[u]·Q[i]); cập nhật bu, bi, P[u], Q[i] (dùng bản sao P[u] cũ khi cập nhật Q[i])
        pass
        ### END CODE HERE ###
        return self

    def predict(self, users, items_):
        ### START CODE HERE ### (≈ 1 dòng)
        return None
        ### END CODE HERE ###

In [ ]:
mf6 = FunkSVD(k=20, lr=0.01, reg=0.1, epochs=40).fit(u_tr, i_tr, y_tr)
rmse6 = float(np.sqrt(np.mean((y_te - np.clip(mf6.predict(u_te, i_te), 1, 5)) ** 2)))
rmse6_train = float(np.sqrt(np.mean((y_tr - mf6.predict(u_tr, i_tr)) ** 2)))
print(f"RMSE train = {rmse6_train:.4f} | RMSE test = {rmse6:.4f}")
check_less(rmse6, 0.625, "tốt hơn baseline μ + b_u + b_i (0.631)")
check_less(rmse6_train, rmse6, "train RMSE thấp hơn test")

## L4 — Debug

**Bài 7.** Đoạn đánh giá popularity dưới đây có 3 lỗi: (1) độ phổ biến đếm trên **toàn bộ** `ratings` (gồm cả test → rò rỉ); (2) không loại phim user **đã thấy** trong train; (3) coi **mọi** phim trong test là "đúng" (kể cả rating 1–3). Sửa (dùng `B`, `seen`, `liked_test` đã có) và lưu `p10_7`.

In [ ]:
### START CODE HERE ### (≈ 8 dòng)
pop_bug = ratings.groupby("item_id").size().reindex(range(n_items), fill_value=0).to_numpy()    # 🐛 (1)
scores = []
for u, g in test.groupby("user_id"):
    top = np.argsort(-pop_bug)[:10]                                                                # 🐛 (2)
    scores.append(M.precision_at_k(top, list(g.item_id), 10))                                      # 🐛 (3)
p10_7 = np.mean(scores)
### END CODE HERE ###
print(f"precision@10 của popularity (đánh giá đúng) = {p10_7:.4f}")

In [ ]:
check_close(p10_7, 0.0993, tol=2e-3, message="precision@10 khớp bài 01")

## L5 — Phân tích

**Bài 8.** **Đánh đổi chính xác ↔ khám phá của PureSVD.** Với $k \in \{2, 5, 10, 20, 50\}$, tính precision@10, **coverage** (tỉ lệ phim từng được gợi ý cho ai đó) và **tail share** (tỉ lệ gợi ý thuộc 80% phim ít phổ biến nhất trong train). Lưu DataFrame `trade8` (index = k). Nhận xét.

In [ ]:
U8, s8, Vt8 = np.linalg.svd(B, full_matrices=False)
head_items = set(np.argsort(-B.sum(0))[: int(0.2 * n_items)])
rows8 = {}
### START CODE HERE ### (≈ 14 dòng)
# với mỗi k: Bk = (U8[:, :k] * s8[:k]) @ Vt8[:k]; precision10(lambda u: Bk[u])
# duyệt user trong liked_test: top-10 (loại phim đã thấy) → gom tập phim đã gợi ý, đếm số gợi ý không thuộc head_items
# rows8[k] = {"precision@10": ..., "coverage": ..., "tail share": ...}
trade8 = None   # pd.DataFrame(rows8).T
### END CODE HERE ###
print(trade8.round(3))

In [ ]:
check_true(int(trade8["precision@10"].idxmax()) <= 10, "độ chính xác tốt nhất ở k nhỏ")
check_greater(trade8.loc[50, "coverage"], trade8.loc[5, "coverage"] + 0.2, "k lớn → độ phủ cao hơn nhiều")
check_greater(trade8.loc[50, "tail share"], trade8.loc[2, "tail share"], "k lớn → gợi ý nhiều phim đuôi dài hơn")

## L6 — Thực tế

**Bài 9.** **Chiến lược cho user mới.** Chọn 60 user có ≥ 20 tương tác (seed 1) làm "user mới": xoá họ khỏi dữ liệu dùng để học; mỗi user chỉ tiết lộ $n$ phim ngẫu nhiên (seed = user_id) trong lịch sử; mục tiêu là đoán **phần còn lại** của lịch sử. So sánh precision@10 của popularity và item-KNN (cosine trên dữ liệu nhị phân của các user còn lại, điểm = tổng tương đồng với $n$ phim đã biết) với $n \in \{1, 3, 5, 10, 15\}$. Lưu `cold9` (DataFrame, index = n, cột "popularity", "item-KNN") và `crossover9` = $n$ nhỏ nhất mà item-KNN tốt hơn popularity. Sau đó viết `switching_scores(known)`: popularity nếu `len(known) < crossover9`, ngược lại item-KNN.

In [ ]:
B_all = np.zeros((n_users, n_items))
B_all[ratings.user_id, ratings.item_id] = 1.0
r9 = np.random.default_rng(1)
new_users = r9.choice(np.flatnonzero(B_all.sum(1) >= 20), 60, replace=False)
B_old = B_all.copy()
B_old[new_users] = 0.0
pop_old = B_old.sum(0)
norm_i = np.sqrt(B_old.sum(0)) + 1e-9
S_old = (B_old.T @ B_old) / np.outer(norm_i, norm_i)
np.fill_diagonal(S_old, 0.0)
### START CODE HERE ### (≈ 16 dòng)
# với mỗi n: với mỗi user mới u: hist = phim đã xem; known = np.random.default_rng(u).choice(hist, n, replace=False)
# target = hist − known; điểm popularity = pop_old, item-KNN = S_old @ b (b = vector nhị phân của known); loại known; precision@10
cold9 = None
crossover9 = None
### END CODE HERE ###


def switching_scores(known):
    ### START CODE HERE ### (≈ 4 dòng)
    return None
    ### END CODE HERE ###


print(cold9.round(3))
print(f"item-KNN bắt đầu vượt popularity khi biết ≥ {crossover9} phim")

In [ ]:
check_greater(cold9.loc[1, "popularity"], cold9.loc[1, "item-KNN"], "biết 1 phim: popularity tốt hơn")
check_greater(cold9.loc[15, "item-KNN"], cold9.loc[15, "popularity"], "biết 15 phim: cá nhân hoá tốt hơn")
check_true(crossover9 in (5, 10, 15), "điểm giao nằm trong khoảng 5–15 phim")
check_true(np.array_equal(switching_scores([1, 2]), pop_old), "ít dữ liệu → popularity")

## L7 — Nâng cao

**Bài 10.** **BPR — Bayesian Personalized Ranking** (Rendle et al., 2009): học trực tiếp **thứ hạng** từ phản hồi ngầm. Với bộ ba $(u, i, j)$ — $i$ đã xem, $j$ chưa xem (lấy mẫu ngẫu nhiên) — tối đa $\ln\sigma(\hat x_{ui} - \hat x_{uj})$ với $\hat x_{ui} = b_i + \mathbf p_u^\top\mathbf q_i$. Gradient với $g = 1 - \sigma(\hat x_{uij}) = \frac{1}{1 + e^{\hat x_{uij}}}$:
$$\mathbf p_u \mathrel{+}= \eta(g(\mathbf q_i - \mathbf q_j) - \lambda\mathbf p_u), \quad \mathbf q_i \mathrel{+}= \eta(g\,\mathbf p_u - \lambda\mathbf q_i), \quad \mathbf q_j \mathrel{+}= \eta(-g\,\mathbf p_u - \lambda\mathbf q_j), \quad b_i \mathrel{+}= \eta(g - \lambda b_i), \quad b_j \mathrel{+}= \eta(-g - \lambda b_j)$$
Cài đặt `bpr(B, k=16, lr=0.05, reg=0.05, epochs=30, seed=0)` → `(P, Q, b)`; mỗi epoch duyệt mọi cặp (u, i) đã xem theo thứ tự ngẫu nhiên, mỗi cặp lấy một $j$ chưa xem. Đánh giá precision@10 (`p10_bpr`) và tỉ lệ bộ ba train được xếp đúng (`auc_train`, ước lượng trên 20 000 bộ ba ngẫu nhiên).

In [ ]:
def bpr(B, k=16, lr=0.05, reg=0.05, epochs=30, seed=0):
    r = np.random.default_rng(seed)
    n_u, n_i = B.shape
    P, Q, b = r.normal(0, 0.1, (n_u, k)), r.normal(0, 0.1, (n_i, k)), np.zeros(n_i)
    users, pos = np.nonzero(B)
    ### START CODE HERE ### (≈ 16 dòng)
    # mỗi epoch: for idx in r.permutation(len(users)): u, i = users[idx], pos[idx]
    #   j = r.integers(n_i) cho tới khi B[u, j] == 0
    #   x = b[i] − b[j] + P[u]·(Q[i] − Q[j]); g = 1 / (1 + exp(x)); cập nhật theo công thức (dùng bản sao P[u] cũ)
    pass
    ### END CODE HERE ###
    return P, Q, b

In [ ]:
P10, Q10, b10 = bpr(B)
p10_bpr = precision10(lambda u: b10 + Q10 @ P10[u])
p10_pop = precision10(lambda u: B.sum(0))
r10 = np.random.default_rng(42)
uu, ii = np.nonzero(B)
pick = r10.integers(len(uu), size=20000)
jj = r10.integers(n_items, size=20000)
valid = B[uu[pick], jj] == 0
x_pos = b10[ii[pick]] + np.sum(P10[uu[pick]] * Q10[ii[pick]], 1)
x_neg = b10[jj] + np.sum(P10[uu[pick]] * Q10[jj], 1)
auc_train = float(np.mean((x_pos > x_neg)[valid]))
print(f"BPR: precision@10 = {p10_bpr:.4f} (popularity {p10_pop:.4f}) | AUC trên bộ ba train = {auc_train:.3f}")
check_greater(auc_train, 0.85, "xếp đúng phần lớn cặp (đã xem > chưa xem) trên train")
check_greater(p10_bpr, p10_pop + 0.01, "BPR vượt popularity")